In [1]:
from pathlib import Path
from typing import Dict, List
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os
import matplotlib.dates as mdates
import itertools

os.chdir("..")
from src.config import FIGURES_DIR



df = pd.read_csv(Path(r"D:\Research\Project_COPD\COPD\data\interim\ALL_PRESCRIPTION_DATA_FILTERED.csv"))
df["Prescription Date"] = pd.to_datetime(df["Prescription Date"], format="%Y%m%d", errors="coerce")
df["Result Numerical Value"] = pd.to_numeric(df["Result Numerical Value"], errors="coerce")

2025-08-25 20:53:19.376 | INFO     | src.config:<module>:11 - PROJ_ROOT path is: D:\Research\Project_COPD\COPD
C:\Users\Shayahn-DKE\AppData\Local\Temp\ipykernel_66960\1104647383.py:15: DtypeWarning: Columns (2,4,10,11,12,13,14) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(Path(r"D:\Research\Project_COPD\COPD\data\interim\ALL_PRESCRIPTION_DATA_FILTERED.csv"))


In [ ]:
# def plot_patient_variables_grid(
#     dfx: pd.DataFrame,
#     patient_id: int,
#     out_dir: Path = Path(FIGURES_DIR / "pft_plots_all_variants"),
#     plot_variable_order: List[str] = None,
#     wanted_measure_indexes: List[str] = None,
#     measure_index_colors: Dict[str, str] = None,
#     marker: str = "o",
#     linewidth: float = 1.6,
# ) -> None:
#     """
#     One figure with seven vertical subplots, one per Variable.
#     Each subplot shows all selected Measurements over time.
#     Each subplot has its own x-ticks, x-label, and title.
#     """

#     if dfx.empty:
#         return

#     dfx = dfx.copy().sort_values("Prescription Date")

#     if plot_variable_order is None:
#         plot_variable_order = ["Meas","Pred","%Pred","%Chg.","Post_Meas","Post_%Pred","Post_%Chg"]
#     if wanted_measure_indexes is None:
#         wanted_measure_indexes = sorted(dfx["Measurement"].dropna().unique().tolist())
#     if measure_index_colors is None:
#         import itertools
#         base = ["C0","C1","C2","C3","C4","C5","C6","C7","C8","C9"]
#         measure_index_colors = {mi: c for mi, c in zip(wanted_measure_indexes, itertools.cycle(base))}

#     nrows = len(plot_variable_order)
#     fig, axes = plt.subplots(nrows=nrows, ncols=1, figsize=(14, 18), sharex=False)
#     if nrows == 1:
#         axes = [axes]

#     handles_all, labels_all = [], []

#     for ax, var in zip(axes, plot_variable_order):
#         dft_var = dfx[dfx["Variable"] == var]
#         # ax.set_xlabel("Prescription Date")
#         ax.set_ylabel(f"{var}")
#         ax.grid(True, linestyle="--", alpha=0.3)

#         any_line = False
#         for mi in wanted_measure_indexes:
#             dft = dft_var[dft_var["Measurement"] == mi]
#             if dft.empty:
#                 continue

#             h, = ax.plot(
#                 dft["Prescription Date"],
#                 dft["Result Numerical Value"],
#                 marker=marker,
#                 linewidth=linewidth,
#                 color=measure_index_colors.get(mi, "black"),
#                 label=mi,
#             )
#             any_line = True

#             if not any(lbl.get_label() == mi for lbl in handles_all):
#                 handles_all.append(h)
#                 labels_all.append(mi)

#         if not any_line:
#             ax.text(0.5, 0.5, "No data", ha="center", va="center",
#                     transform=ax.transAxes, alpha=0.6)

#         # Force every Prescription Date to appear on x-axis
#         # ax.set_xticks(dft_var["Prescription Date"].unique())
#         # ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y-%m-%d"))

#         # ⬇️ Show full date format (YYYY-MM-DD)
#         ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y-%m-%d"))
#         ax.xaxis.set_major_locator(mdates.AutoDateLocator())

#         # Rotate tick labels for readability
#         for tick in ax.get_xticklabels():
#             tick.set_rotation(30)
#             tick.set_ha("right")

        
#     # ✅ Only set xlabel for the last subplot
#     axes[-1].set_xlabel("Prescription Date")

#     fig.suptitle(f"Patient {patient_id} — All Variables", y=0.995, fontsize=14)

#     if handles_all:
#         fig.legend(
#             handles_all, labels_all,
#             loc="lower center", bbox_to_anchor=(0.5, 0.0),
#             ncol=min(5, len(labels_all)), fontsize=10, frameon=False
#         )
#         fig.subplots_adjust(bottom=0.12, top=0.95, hspace=0.5)  # ⬅ increase bottom for long dates
#     else:
#         fig.subplots_adjust(top=0.95, hspace=0.5)

#     out_dir = Path(out_dir)
#     out_dir.mkdir(parents=True, exist_ok=True)
#     out_name = f"patient_{patient_id}_variables_grid.png"
#     fig.savefig(out_dir / out_name, dpi=300, bbox_inches="tight")
#     plt.close(fig)

In [ ]:
# patient_ids = [886482, 1207865, 1452945, 611957, 965594, 5665, 7429, 29903]
# for pid in patient_ids:
#     dfx = df[(df["Patient Number"] == pid)]
#     plot_patient_variables_grid(dfx, pid)

In [2]:
import pandas as pd
import numpy as np
from typing import Tuple, Dict

In [2]:
def detect_anomalies(dfx: pd.DataFrame,
                     date_col: str = "Prescription Date",
                     value_col: str = "Result Numerical Value",
                     measure_col: str = "Measurement",
                     variable_col: str = "Variable",
                     z_thresh: float = 3.5,
                     jump_thresh_per_month: Dict[str, float] | None = None
                    ) -> Tuple[pd.DataFrame, str]:
    """
    Returns:
      flagged_df (same rows as dfx, with anomaly columns)
      title_tag  (e.g., 'Missing value+Out of valid range+MAD+Jump' for this patient)

    Flags:
      - Anomaly_Missing: value == 0 or NaN
      - Anomaly_Range: outside clinical ranges
          FEV1 (Meas/Post_Meas): 0.2–10.0
          FVC  (Meas/Post_Meas): 0.3–10.0
          DLCO (Meas/Post_Meas): 0.3–50
          FEV1/FVC ratio (Meas/Post_Meas): 0.2–1.2
          %Pred ( %Pred/Post_%Pred ): 0–200
      - Outlier_MAD: |robust_z| > z_thresh (per-measurement series)
      - Outlier_Jump: month-normalized step exceeds measurement threshold
    """
    jump_thresh_per_month = jump_thresh_per_month or {
        "FEV1": 0.30, "FVC": 0.40, "DLCO": 3.0, "FEV1/FVC": 0.08, "DLCO/VA": 0.60,
    }
    abs_vars   = {"Meas", "Post_Meas"}
    perc_vars  = {"%Pred", "Post_%Pred"}

    df2 = dfx.copy()

    # Ensure numeric + datetime for calculations
    v = pd.to_numeric(df2[value_col], errors="coerce")
    dt = pd.to_datetime(df2[date_col], errors="coerce")
    df2["_val_"] = v
    df2["_date_"] = dt

    # 1) Missing (as requested: treat 0 as missing; also NaN is missing)
    df2["Anomaly_Missing"] = v.isna() | (v == 0)

    # 2) Valid ranges
    rng_flag = pd.Series(False, index=df2.index)

    def _violate(series_mask, low, high):
        if not series_mask.any(): 
            return pd.Series(False, index=df2.index)
        s = v.where(series_mask)
        return (s < low) | (s > high)

    # Absolute measurements (Meas/Post_Meas)
    m = df2[measure_col].astype(str)
    var = df2[variable_col].astype(str)

    mask_abs = var.isin(abs_vars)
    rng_flag |= _violate(mask_abs & (m == "FEV1"), 0.2, 10.0)
    rng_flag |= _violate(mask_abs & (m == "FVC"),  0.3, 10.0)
    rng_flag |= _violate(mask_abs & (m == "DLCO"), 0.3, 50.0)

    # Ratio FEV1/FVC (absolute)
    rng_flag |= _violate(mask_abs & (m == "FEV1/FVC"), 0.2, 1.2)

    # Percent predicted
    mask_perc = var.isin(perc_vars)
    rng_flag |= _violate(mask_perc, 0.0, 200.0)

    df2["Anomaly_Range"] = rng_flag.fillna(False)

    # 3) Outliers (MAD + Jump) per measurement series
    out_mad  = pd.Series(False, index=df2.index)
    out_jump = pd.Series(False, index=df2.index)

    for mi, g in df2.groupby(measure_col, dropna=False):
        g = g.sort_values("_date_")
        vv = g["_val_"]
        dd = g["_date_"]

        # MAD
        med = vv.median()
        mad = float(np.median(np.abs(vv - med))) if len(vv) else 0.0
        mad = mad if mad > 0 else 1e-9
        robust_z = 0.6745 * (vv - med) / mad
        out_mad.loc[g.index] = robust_z.abs() > z_thresh

        # Jump per month
        dv = vv.diff().abs()
        dt_days = dd.diff().dt.days
        dt_days = dt_days.where(dt_days > 0, 1)  # avoid 0/NaN/<=0
        months = dt_days / 30.0
        thr = jump_thresh_per_month.get(str(mi), np.inf)
        out_jump.loc[g.index] = (dv / months) > thr

    df2["Outlier_MAD"]  = out_mad.fillna(False)
    df2["Outlier_Jump"] = out_jump.fillna(False)
    df2["Outlier"]      = df2["Outlier_MAD"] | df2["Outlier_Jump"]

    # Compose row-wise tags
    def _row_tags(row):
        tags = []
        if row["Anomaly_Missing"]: tags.append("Missing value")
        if row["Anomaly_Range"]:   tags.append("Out of valid range")
        if row["Outlier_MAD"]:     tags.append("MAD")
        if row["Outlier_Jump"]:    tags.append("Jump")
        return " | ".join(tags)

    df2["Anomaly_Tags"] = df2.apply(_row_tags, axis=1)
    # Short tag for tight annotations
    df2["Anomaly_Tags_Short"] = (df2["Anomaly_Tags"]
                                 .str.replace("Missing value", "Miss", regex=False)
                                 .str.replace("Out of valid range", "Range", regex=False))

    df2["Anomaly_Any"] = df2[["Anomaly_Missing","Anomaly_Range","Outlier_MAD","Outlier_Jump"]].any(axis=1)

    # Build patient-level title tag
    present = []
    if df2["Anomaly_Missing"].any(): present.append("Missing value")
    if df2["Anomaly_Range"].any():   present.append("Out of valid range")
    if df2["Outlier_MAD"].any():     present.append("MAD")
    if df2["Outlier_Jump"].any():    present.append("Jump")
    title_tag = "+".join(present)

    # Clean temp cols for plotting (keep dates numeric as original)
    df2 = df2.drop(columns=["_val_", "_date_"])

    return df2, title_tag


NameError: name 'Tuple' is not defined

In [ ]:
# def plot_patient_variables_grid(
#     dfx: pd.DataFrame,
#     patient_id: int,
#     out_dir: Path = Path(FIGURES_DIR / "pft_plots_all_variants"),
#     plot_variable_order: List[str] = None,
#     wanted_measure_indexes: List[str] = None,
#     measure_index_colors: Dict[str, str] = None,
#     marker: str = "o",
#     linewidth: float = 1.6,
#     *,
#     title_suffix: str = "",
# ) -> None:
#     """
#     One figure with seven vertical subplots, one per Variable.
#     Each subplot shows all selected Measurements over time.
#     Each subplot has its own x-ticks and only the last subplot has x-label.
#     Draw colored circle outlines for anomalies (no text on plot), and add a
#     bottom legend explaining circle colors:
#         - Missing Value (tab:orange)
#         - Out of Valid Range (tab:red)
#         - MAD (tab:purple)
#         - Jump (tab:green)

#     Expects (if available) boolean columns per row:
#       'Anomaly_Missing', 'Anomaly_Range', 'Outlier_MAD', 'Outlier_Jump'
#     (If not present, the function still plots without anomaly circles.)
#     """
#     import itertools
#     import matplotlib.dates as mdates
#     from matplotlib.lines import Line2D

#     if dfx.empty:
#         return

#     dfx = dfx.copy()
#     # ensure datetime & sort
#     dfx["Prescription Date"] = pd.to_datetime(dfx["Prescription Date"], errors="coerce")
#     dfx = dfx.sort_values("Prescription Date")

#     if plot_variable_order is None:
#         plot_variable_order = ["Meas","Pred","%Pred","%Chg.","Post_Meas","Post_%Pred","Post_%Chg"]
#     if wanted_measure_indexes is None:
#         wanted_measure_indexes = sorted(dfx["Measurement"].dropna().unique().tolist())
#     if measure_index_colors is None:
#         base = ["C0","C1","C2","C3","C4","C5","C6","C7","C8","C9"]
#         measure_index_colors = {mi: c for mi, c in zip(wanted_measure_indexes, itertools.cycle(base))}

#     # Colors for anomaly circles
#     ANOM_COLORS = {
#         "Missing": "tab:orange",
#         "Range":   "tab:red",
#         "MAD":     "tab:purple",
#         "Jump":    "tab:green",
#     }
#     have_anom_cols = all(col in dfx.columns for col in
#                          ["Anomaly_Missing","Anomaly_Range","Outlier_MAD","Outlier_Jump"])

#     nrows = len(plot_variable_order)
#     fig, axes = plt.subplots(nrows=nrows, ncols=1, figsize=(14, 18), sharex=False)
#     if nrows == 1:
#         axes = [axes]

#     handles_all, labels_all = [], []

#     for ax, var in zip(axes, plot_variable_order):
#         dft_var = dfx[dfx["Variable"] == var]
#         ax.set_ylabel(f"{var}")
#         ax.grid(True, linestyle="--", alpha=0.3)

#         any_line = False
#         for mi in wanted_measure_indexes:
#             dft = dft_var[dft_var["Measurement"] == mi]
#             if dft.empty:
#                 continue

#             # main line
#             h, = ax.plot(
#                 dft["Prescription Date"],
#                 dft["Result Numerical Value"],
#                 marker=marker,
#                 linewidth=linewidth,
#                 color=measure_index_colors.get(mi, "black"),
#                 label=mi,
#                 zorder=2,
#             )
#             any_line = True
#             if not any(lbl.get_label() == mi for lbl in handles_all):
#                 handles_all.append(h)
#                 labels_all.append(mi)

#             # anomaly circles (no text)
#             if have_anom_cols and not dft.empty:                
#                 if dft["Outlier_MAD"].any():
#                     bad = dft[dft["Outlier_MAD"]]
#                     ax.scatter(
#                         bad["Prescription Date"], bad["Result Numerical Value"],
#                         s=70, facecolors="none", edgecolors=ANOM_COLORS["MAD"],
#                         linewidths=1.8, zorder=4
#                     )
#                 if dft["Outlier_Jump"].any():
#                     bad = dft[dft["Outlier_Jump"]]
#                     ax.scatter(
#                         bad["Prescription Date"], bad["Result Numerical Value"],
#                         s=70, facecolors="none", edgecolors=ANOM_COLORS["Jump"],
#                         linewidths=1.8, zorder=4
#                     )                
#                 if dft["Anomaly_Range"].any():
#                     bad = dft[dft["Anomaly_Range"]]
#                     ax.scatter(
#                         bad["Prescription Date"], bad["Result Numerical Value"],
#                         s=70, facecolors="none", edgecolors=ANOM_COLORS["Range"],
#                         linewidths=1.8, zorder=4
#                     )
#                 if dft["Anomaly_Missing"].any():
#                     bad = dft[dft["Anomaly_Missing"]]
#                     ax.scatter(
#                         bad["Prescription Date"], bad["Result Numerical Value"],
#                         s=70, facecolors="none", edgecolors=ANOM_COLORS["Missing"],
#                         linewidths=1.8, zorder=4
#                     )

#         if not any_line:
#             ax.text(0.5, 0.5, "No data", ha="center", va="center",
#                     transform=ax.transAxes, alpha=0.6)

#         # full date format on each subplot
#         ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y-%m-%d"))
#         ax.xaxis.set_major_locator(mdates.AutoDateLocator())
#         for tick in ax.get_xticklabels():
#             tick.set_rotation(30)
#             tick.set_ha("right")

#     # only bottom subplot x-label
#     axes[-1].set_xlabel("Prescription Date")

#     # figure title (optional suffix)
#     fig.suptitle(
#         f"Patient {patient_id} — All Variables{(' — ' + title_suffix) if title_suffix else ''}",
#         y=0.995, fontsize=14
#     )

#     # legend for measurement lines
#     if handles_all:
#         fig.legend(
#             handles_all, labels_all,
#             loc="lower center", bbox_to_anchor=(0.5, 0.0),
#             ncol=min(5, len(labels_all)), fontsize=10, frameon=False
#         )
#         fig.subplots_adjust(bottom=0.12, top=0.95, hspace=0.5)
#     else:
#         fig.subplots_adjust(top=0.95, hspace=0.5)

#     # anomaly legend (explainer) — added below the first legend
#     anom_handles = [
#         Line2D([0],[0], marker='o', linestyle='None', markersize=8,
#                markerfacecolor='none', markeredgecolor=ANOM_COLORS["Missing"], label="Missing Value"),
#         Line2D([0],[0], marker='o', linestyle='None', markersize=8,
#                markerfacecolor='none', markeredgecolor=ANOM_COLORS["Range"], label="Out of Valid Range"),
#         Line2D([0],[0], marker='o', linestyle='None', markersize=8,
#                markerfacecolor='none', markeredgecolor=ANOM_COLORS["MAD"], label="MAD"),
#         Line2D([0],[0], marker='o', linestyle='None', markersize=8,
#                markerfacecolor='none', markeredgecolor=ANOM_COLORS["Jump"], label="Jump"),
#     ]
#     fig.legend(
#         anom_handles, [h.get_label() for h in anom_handles],
#         loc="lower center", bbox_to_anchor=(0.5, -0.06),
#         ncol=4, fontsize=9, frameon=False
#     )
#     # make room for the extra legend row
#     fig.subplots_adjust(bottom=0.20)

#     out_dir = Path(out_dir)
#     out_dir.mkdir(parents=True, exist_ok=True)
#     out_name = f"patient_{patient_id}_variables_grid.png"
#     fig.savefig(out_dir / out_name, dpi=300, bbox_inches="tight")
#     plt.close(fig)


In [ ]:
# def plot_patient_variables_grid(
#     dfx: pd.DataFrame,
#     patient_id: int,
#     out_dir: Path = Path(FIGURES_DIR / "pft_plots_all_variants"),
#     plot_variable_order: List[str] = None,
#     wanted_measure_indexes: List[str] = None,
#     measure_index_colors: Dict[str, str] = None,
#     marker: str = "o",
#     linewidth: float = 1.6,
#     *,
#     title_suffix: str = "",
# ) -> None:
#     """
#     One figure with seven vertical subplots, one per Variable.
#     Each subplot shows all selected Measurements over time.
#     Each subplot has its own x-ticks and only the last subplot has x-label.
#     Draw colored circle outlines for anomalies (no text on plot), and add a
#     bottom legend explaining circle colors:
#         - Missing Value (tab:orange)
#         - Out of Valid Range (tab:red)
#         - MAD (tab:purple)
#         - Jump (tab:green)

#     Expects (if available) boolean columns per row:
#       'Anomaly_Missing', 'Anomaly_Range', 'Outlier_MAD', 'Outlier_Jump'
#     (If not present, the function still plots without anomaly circles.)
#     """
#     import itertools
#     import matplotlib.dates as mdates
#     from matplotlib.lines import Line2D

#     if dfx.empty:
#         return

#     dfx = dfx.copy()
#     # ensure datetime & sort
#     dfx["Prescription Date"] = pd.to_datetime(dfx["Prescription Date"], errors="coerce")
#     dfx = dfx.sort_values("Prescription Date")

#     if plot_variable_order is None:
#         # plot_variable_order = ["Meas","Pred","%Pred","%Chg.","Post_Meas","Post_%Pred","Post_%Chg"]
#         plot_variable_order = ["Meas", "%Pred","%Chg.","Post_Meas","Post_%Pred","Post_%Chg"]
#     if wanted_measure_indexes is None:
#         wanted_measure_indexes = sorted(dfx["Measurement"].dropna().unique().tolist())
#     if measure_index_colors is None:
#         base = ["C0","C1","C2","C3","C4","C5","C6","C7","C8","C9"]
#         measure_index_colors = {mi: c for mi, c in zip(wanted_measure_indexes, itertools.cycle(base))}

#     # Colors for anomaly circles
#     # ANOM_COLORS = {"Missing": "tab:orange","Range": "tab:red", "MAD": "tab:purple", "Jump": "tab:green"}
#     ANOM_COLORS = {
#         "Missing": "#FFB300",   # amber (not the default 'C1' orange)
#         "Range":   "#8B0000",   # dark red
#         "MAD":     "#000000",   # black
#         "Jump":    "#00BFA6",   # teal
#     }
#     have_anom_cols = all(col in dfx.columns for col in
#                          ["Anomaly_Missing","Anomaly_Range","Outlier_MAD","Outlier_Jump"])

#     nrows = len(plot_variable_order)
#     # fig, axes = plt.subplots(nrows=nrows, ncols=1, figsize=(14, 18), sharex=False)
#     fig, axes = plt.subplots(nrows=nrows, ncols=1, figsize=(20, 2*nrows), sharex=False)
#     if nrows == 1:
#         axes = [axes]

#     handles_all, labels_all = [], []

#     for ax, var in zip(axes, plot_variable_order):
#         dft_var = dfx[dfx["Variable"] == var]
#         ax.set_ylabel(f"{var}")
#         ax.grid(True, linestyle="--", alpha=0.3)

#         any_line = False
#         for mi in wanted_measure_indexes:
#             dft = dft_var[dft_var["Measurement"] == mi]
#             if dft.empty:
#                 continue

#             # main line
#             h, = ax.plot(
#                 dft["Prescription Date"],
#                 dft["Result Numerical Value"],
#                 marker=marker,
#                 linewidth=linewidth,
#                 color=measure_index_colors.get(mi, "black"),
#                 label=mi,
#                 zorder=2,
#             )
#             any_line = True
#             if not any(lbl.get_label() == mi for lbl in handles_all):
#                 handles_all.append(h)
#                 labels_all.append(mi)

#             # anomaly circles (no text)
#             if have_anom_cols and not dft.empty:
#                 anom_specs = [
#                     ("MAD",     "Outlier_MAD"),
#                     ("Jump",    "Outlier_Jump"),
#                     ("Range",   "Anomaly_Range"),
#                     ("Missing", "Anomaly_Missing"),
#                 ]
#                 for key, col in anom_specs:
#                     if col in dft.columns and dft[col].any():
#                         bad = dft[dft[col]]
#                         ax.scatter(
#                             bad["Prescription Date"], bad["Result Numerical Value"],
#                             s=80, marker="s",            # ← square (rectangle) outline
#                             facecolors="none", edgecolors=ANOM_COLORS[key],
#                             linewidths=1.8, zorder=4
#                         )

#         if not any_line:
#             ax.text(0.5, 0.5, "No data", ha="center", va="center",
#                     transform=ax.transAxes, alpha=0.6)

#         # full date format on each subplot
#         ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y-%m-%d"))
#         ax.xaxis.set_major_locator(mdates.AutoDateLocator())
#         for tick in ax.get_xticklabels():
#             tick.set_rotation(30)
#             tick.set_ha("right")

#     # only bottom subplot x-label
#     axes[-1].set_xlabel("Prescription Date")

#     # figure title (optional suffix)
#     fig.suptitle(
#         f"Patient {patient_id} — All Variables{(' — ' + title_suffix) if title_suffix else ''}",
#         y=0.995, fontsize=14
#     )

#     # -------- LEGENDS (measurement row, then one anomaly per line) --------
#     # 1) Measurement legend: single row
#     if handles_all:
#         fig.legend(
#             handles_all, labels_all,
#             loc="lower center", bbox_to_anchor=(0.5, 0.08),
#             ncol=len(labels_all), fontsize=9, frameon=False,
#             handlelength=2.0, handletextpad=0.6, columnspacing=1.2
#         )

#     # 2) One line per anomaly (stacked)
#     anom_specs = [
#         ("Missing Value: Value is zero.", ANOM_COLORS["Missing"]),
#         ("Out of Valid Range: FEV1 & FVC (Meas/Post_Meas) 0.2-10.0; DLCO 0.3-50; FEV1/FVC 0.2-1.2; %Pred 0-200.", ANOM_COLORS["Range"]),
#         ("MAD: |robust_z| > z_thresh (3.5)", ANOM_COLORS["MAD"]),
#         ("Jump: Δ/month > thresholds (FEV1 0.30, FVC 0.40, DLCO 3.0, FEV1/FVC 0.08, DLCO/VA 0.60).", ANOM_COLORS["Jump"]),
#     ]
#     # y0, dy = 0.055, 0.028   # starting y and spacing between lines
#     y0, dy = 0.07, 0.01   # starting y and spacing between lines
#     for i, (lab, col) in enumerate(anom_specs):
#         h = Line2D([0],[0], marker='s', linestyle='None', markersize=8,
#                    markerfacecolor='none', markeredgecolor=col, label=lab)
#         fig.legend(
#             [h], [lab],
#             loc="lower center", bbox_to_anchor=(0.5, y0 - i*dy),
#             ncol=1, fontsize=9, frameon=False, handlelength=1.2
#         )

#     # room for 1 (measurements) + 4 (anomalies) lines
#     fig.subplots_adjust(bottom=0.15, top=0.95, hspace=0.5)

#     out_dir = Path(out_dir)
#     out_dir.mkdir(parents=True, exist_ok=True)
#     out_name = f"patient_{patient_id}_variables_grid.png"
#     fig.savefig(out_dir / out_name, dpi=300, bbox_inches="tight")
#     plt.close(fig)

# plot_patient_variables_grid(dfx_flagged, pid, title_suffix=title_tag)

In [ ]:
# def plot_patient_variables_grid(
#     dfx: pd.DataFrame,
#     patient_id: int,
#     out_dir: Path = Path(FIGURES_DIR / "pft_plots_all_variants"),
#     plot_variable_order: List[str] = None,
#     wanted_measure_indexes: List[str] = None,
#     measure_index_colors: Dict[str, str] = None,
#     marker: str = "o",
#     linewidth: float = 1.6,
#     *,
#     title_suffix: str = "",
# ) -> None:
#     """
#     Create one figure with vertical subplots (one per Variable).
#     - Plots selected Measurements over time (per subplot).
#     - Uses full YYYY-MM-DD dates on x-axis; only bottom subplot has x-label.
#     - Overlays anomaly markers as hollow squares (no text labels).
#     - Adds measurement legend (single row) and stacked anomaly legend lines.

#     Expects anomaly columns if available:
#       'Anomaly_Missing', 'Anomaly_Range', 'Outlier_MAD', 'Outlier_Jump'
#     The plot renders fine even if they are absent.
#     """
#     import itertools
#     import matplotlib.dates as mdates
#     from matplotlib.lines import Line2D

#     if dfx.empty:
#         return

#     # ---- constants / basic prep ----
#     DATE_COL  = "Prescription Date"
#     VALUE_COL = "Result Numerical Value"
#     VAR_COL   = "Variable"
#     MEAS_COL  = "Measurement"

#     dfx = dfx.copy()
#     dfx[DATE_COL] = pd.to_datetime(dfx[DATE_COL], errors="coerce")
#     dfx = dfx.sort_values(DATE_COL)

#     if plot_variable_order is None:
#         plot_variable_order = ["Meas", "%Pred", "%Chg.", "Post_Meas", "Post_%Pred", "Post_%Chg"]
#     if wanted_measure_indexes is None:
#         wanted_measure_indexes = sorted(dfx[MEAS_COL].dropna().unique().tolist())
#     if measure_index_colors is None:
#         base = ["C0","C1","C2","C3","C4","C5","C6","C7","C8","C9"]
#         measure_index_colors = {mi: c for mi, c in zip(wanted_measure_indexes, itertools.cycle(base))}

#     # anomaly colors (distinct from typical Matplotlib defaults)
#     ANOM_COLORS = {
#         "Missing": "#FFB300",  # amber
#         "Range":   "#8B0000",  # dark red
#         "MAD":     "#000000",  # black
#         "Jump":    "#00BFA6",  # teal
#     }
#     have_anom_cols = all(
#         col in dfx.columns
#         for col in ["Anomaly_Missing", "Anomaly_Range", "Outlier_MAD", "Outlier_Jump"]
#     )

#     # ---- figure & axes ----
#     nrows = len(plot_variable_order)
#     fig, axes = plt.subplots(nrows=nrows, ncols=1, figsize=(20, 2 * nrows), sharex=False)
#     if nrows == 1:
#         axes = [axes]

#     handles_all, labels_all = [], []

#     # ---- plotting per variable ----
#     for ax, var in zip(axes, plot_variable_order):
#         dft_var = dfx[dfx[VAR_COL] == var]
#         ax.set_ylabel(var)
#         ax.grid(True, linestyle="--", alpha=0.3)

#         any_line = False
#         for mi in wanted_measure_indexes:
#             dft = dft_var[dft_var[MEAS_COL] == mi]
#             if dft.empty:
#                 continue

#             # measurement series (circles)
#             h, = ax.plot(
#                 dft[DATE_COL],
#                 dft[VALUE_COL],
#                 marker=marker, linewidth=linewidth,
#                 color=measure_index_colors.get(mi, "black"),
#                 label=mi, zorder=2,
#             )
#             any_line = True
#             if not any(lbl.get_label() == mi for lbl in handles_all):
#                 handles_all.append(h)
#                 labels_all.append(mi)

#             # anomaly overlays (squares) — only if anomaly cols exist
#             if have_anom_cols:
#                 anom_specs = [
#                     ("MAD",     "Outlier_MAD"),
#                     ("Jump",    "Outlier_Jump"),
#                     ("Range",   "Anomaly_Range"),
#                     ("Missing", "Anomaly_Missing"),
#                 ]
#                 for key, col in anom_specs:
#                     if col in dft.columns and dft[col].any():
#                         bad = dft[dft[col]]
#                         ax.scatter(
#                             bad[DATE_COL], bad[VALUE_COL],
#                             s=80, marker="s",
#                             facecolors="none", edgecolors=ANOM_COLORS[key],
#                             linewidths=1.8, zorder=4
#                         )

#         if not any_line:
#             ax.text(0.5, 0.5, "No data", ha="center", va="center",
#                     transform=ax.transAxes, alpha=0.6)

#         # x-axis as full date per subplot
#         ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y-%m-%d"))
#         ax.xaxis.set_major_locator(mdates.AutoDateLocator())
#         for tick in ax.get_xticklabels():
#             tick.set_rotation(30)
#             tick.set_ha("right")

#     # x-label only on bottom subplot
#     axes[-1].set_xlabel("Prescription Date")

#     # title (with optional suffix)
#     fig.suptitle(
#         f"Patient {patient_id} — All Variables{(' — ' + title_suffix) if title_suffix else ''}",
#         y=0.995, fontsize=14
#     )

#     # -------- LEGENDS (measurement row, then one anomaly per line) --------
#     # 1) Measurement legend: single row
#     if handles_all:
#         fig.legend(
#             handles_all, labels_all,
#             loc="lower center", bbox_to_anchor=(0.5, 0.08),
#             ncol=len(labels_all), fontsize=9, frameon=False,
#             handlelength=2.0, handletextpad=0.6, columnspacing=1.2
#         )

#     # 2) One line per anomaly (stacked)
#     anom_specs = [
#         ("Missing Value: Value is zero.", ANOM_COLORS["Missing"]),
#         ("Out of Valid Range: FEV1 & FVC (Meas/Post_Meas) 0.2-10.0; DLCO 0.3-50; FEV1/FVC 0.2-1.2; %Pred 0-200.", ANOM_COLORS["Range"]),
#         ("MAD: |robust_z| > z_thresh (3.5)", ANOM_COLORS["MAD"]),
#         ("Jump: Δ/month > thresholds (FEV1 0.30, FVC 0.40, DLCO 3.0, FEV1/FVC 0.08, DLCO/VA 0.60).", ANOM_COLORS["Jump"]),
#     ]
#     y0, dy = 0.07, 0.01   # starting y and spacing between lines
#     for i, (lab, col) in enumerate(anom_specs):
#         h = Line2D([0],[0], marker='s', linestyle='None', markersize=8,
#                    markerfacecolor='none', markeredgecolor=col, label=lab)
#         fig.legend(
#             [h], [lab],
#             loc="lower center", bbox_to_anchor=(0.5, y0 - i*dy),
#             ncol=1, fontsize=9, frameon=False, handlelength=1.2
#         )

#     # room for 1 (measurements) + 4 (anomalies) lines
#     fig.subplots_adjust(bottom=0.19, top=0.95, hspace=0.5)

#     # ---- save ----
#     out_dir = Path(out_dir)
#     out_dir.mkdir(parents=True, exist_ok=True)
#     out_name = f"patient_{patient_id}_variables_grid.png"
#     fig.savefig(out_dir / out_name, dpi=300, bbox_inches="tight")
#     plt.close(fig)

# plot_patient_variables_grid(dfx_flagged, pid, title_suffix=title_tag)

In [3]:
def plot_patient_variables_grid(
    dfx: pd.DataFrame,
    patient_id: int,
    out_dir: Path = Path(FIGURES_DIR / "pft_plots_all_variants"),
    plot_variable_order: List[str] = None,
    wanted_measure_indexes: List[str] = None,
    measure_index_colors: Dict[str, str] = None,
    marker: str = "o",
    linewidth: float = 1.6,
    plot_anomaly: bool = True,
    *,
    title_suffix: str = "",
) -> None:
    """
    Create one figure with vertical subplots (one per Variable).
    - Plots selected Measurements over time (per subplot).
    - Uses full YYYY-MM-DD dates on x-axis; only bottom subplot has x-label.
    - Overlays anomaly markers as hollow squares (no text labels).
    - Adds measurement legend (single row) and stacked anomaly legend lines.

    Expects anomaly columns if available:
      'Anomaly_Missing', 'Anomaly_Range', 'Outlier_MAD', 'Outlier_Jump'
    The plot renders fine even if they are absent.
    """
    import itertools
    import matplotlib.dates as mdates
    from matplotlib.lines import Line2D

    if dfx.empty:
        return

    # ---- constants / basic prep ----
    DATE_COL  = "Prescription Date"
    VALUE_COL = "Result Numerical Value"
    VAR_COL   = "Variable"
    MEAS_COL  = "Measurement"

    dfx = dfx.copy()
    dfx[DATE_COL] = pd.to_datetime(dfx[DATE_COL], errors="coerce")
    dfx = dfx.sort_values(DATE_COL)

    if plot_variable_order is None:
        plot_variable_order = ["Meas", "%Pred", "%Chg.", "Post_Meas", "Post_%Pred", "Post_%Chg"]
    if wanted_measure_indexes is None:
        wanted_measure_indexes = sorted(dfx[MEAS_COL].dropna().unique().tolist())
    if measure_index_colors is None:
        base = ["C0","C1","C2","C3","C4","C5","C6","C7","C8","C9"]
        measure_index_colors = {mi: c for mi, c in zip(wanted_measure_indexes, itertools.cycle(base))}

    # anomaly colors (distinct from typical Matplotlib defaults)
    ANOM_COLORS = {
        "Missing": "#FFB300",  # amber
        "Range":   "#8B0000",  # dark red
        "MAD":     "#000000",  # black
        "Jump":    "#00BFA6",  # teal
    }
    have_anom_cols = all(
        col in dfx.columns
        for col in ["Anomaly_Missing", "Anomaly_Range", "Outlier_MAD", "Outlier_Jump"]
    )

    # ---- figure & axes ----
    nrows = len(plot_variable_order)
    fig, axes = plt.subplots(nrows=nrows, ncols=1, figsize=((16*2.3*nrows) / 9, 2.3*nrows), sharex=False)
    if nrows == 1:
        axes = [axes]

    handles_all, labels_all = [], []

    # ---- plotting per variable ----
    for ax, var in zip(axes, plot_variable_order):
        dft_var = dfx[dfx[VAR_COL] == var]
        ax.set_ylabel(var)
        ax.grid(True, linestyle="--", alpha=0.3)

        any_line = False
        for mi in wanted_measure_indexes:
            dft = dft_var[dft_var[MEAS_COL] == mi]
            if dft.empty:
                continue

            # measurement series (circles)
            h, = ax.plot(
                dft[DATE_COL],
                dft[VALUE_COL],
                marker=marker, linewidth=linewidth,
                color=measure_index_colors.get(mi, "black"),
                label=mi, zorder=2,
            )
            any_line = True
            if not any(lbl.get_label() == mi for lbl in handles_all):
                handles_all.append(h)
                labels_all.append(mi)

            # anomaly overlays (squares) — only if anomaly cols exist
            if plot_anomaly and have_anom_cols:
                anom_specs = [
                    ("MAD",     "Outlier_MAD"),
                    ("Jump",    "Outlier_Jump"),
                    ("Range",   "Anomaly_Range"),
                    ("Missing", "Anomaly_Missing"),
                ]
                for key, col in anom_specs:
                    if col in dft.columns and dft[col].any():
                        bad = dft[dft[col]]
                        ax.scatter(
                            bad[DATE_COL], bad[VALUE_COL],
                            s=80, marker="s",
                            facecolors="none", edgecolors=ANOM_COLORS[key],
                            linewidths=1.8, zorder=4
                        )

        if not any_line:
            ax.text(0.5, 0.5, "No data", ha="center", va="center",
                    transform=ax.transAxes, alpha=0.6)

        # x-axis as full date per subplot
        ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y-%m-%d"))
        ax.xaxis.set_major_locator(mdates.AutoDateLocator())
        for tick in ax.get_xticklabels():
            tick.set_rotation(30)
            tick.set_ha("right")

    # x-label only on bottom subplot
    axes[-1].set_xlabel("Prescription Date")

    # title (with optional suffix)
    fig.suptitle(
        f"Patient {patient_id} — All Variables{(' — ' + title_suffix) if title_suffix else ''}",
        y=0.995, fontsize=14
    )

    # -------- LEGENDS (measurement row, then one anomaly per line) --------
    # 1) Measurement legend: single row
    if handles_all:
        fig.legend(
            handles_all, labels_all,
            loc="lower center", bbox_to_anchor=(0.5, 0.08),
            ncol=len(labels_all), fontsize=9, frameon=False,
            handlelength=2.0, handletextpad=0.6, columnspacing=1.2
        )

    # 2) One line per anomaly (stacked)
    anom_specs = [
        ("Missing Value: Value is zero.", ANOM_COLORS["Missing"]),
        ("Out of Valid Range: FEV1 & FVC (Meas/Post_Meas) 0.2-10.0; DLCO 0.3-50; FEV1/FVC 0.2-1.2; %Pred 0-200.", ANOM_COLORS["Range"]),
        ("MAD: |robust_z| > z_thresh (3.5)", ANOM_COLORS["MAD"]),
        ("Jump: Δ/month > thresholds (FEV1 0.30, FVC 0.40, DLCO 3.0, FEV1/FVC 0.08, DLCO/VA 0.60).", ANOM_COLORS["Jump"]),
    ]
    y0, dy = 0.07, 0.01   # starting y and spacing between lines
    for i, (lab, col) in enumerate(anom_specs):
        h = Line2D([0],[0], marker='s', linestyle='None', markersize=8,
                   markerfacecolor='none', markeredgecolor=col, label=lab)
        fig.legend(
            [h], [lab],
            loc="lower center", bbox_to_anchor=(0.5, y0 - i*dy),
            ncol=1, fontsize=9, frameon=False, handlelength=1.2
        )

    # room for 1 (measurements) + 4 (anomalies) lines
    fig.subplots_adjust(bottom=0.16, top=0.95, hspace=0.5)

    # ---- save ----
    out_dir = Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)
    out_name = f"patient_{patient_id}_variables_grid.png"
    fig.savefig(out_dir / out_name, dpi=300, bbox_inches="tight")
    plt.close(fig)

# plot_patient_variables_grid(dfx_flagged, pid, title_suffix=title_tag)

In [6]:
patient_ids = [886482, 1207865, 1452945, 611957, 965594, 5665, 7429, 29903]

for pid in patient_ids:
    dfx = df[df["Patient Number"] == pid].copy()
    dfx_flagged, title_tag = detect_anomalies(dfx)            # ← one function
    plot_patient_variables_grid(dfx_flagged, pid, title_suffix=title_tag)

In [ ]:
import os
from concurrent.futures import ProcessPoolExecutor, as_completed
import matplotlib
matplotlib.use("Agg")  # headless, safe in child processes

def _plot_one(args):
    pid, dfx = args

    # Prevent thread over-subscription inside each process
    os.environ.setdefault("MKL_NUM_THREADS", "1")
    os.environ.setdefault("OPENBLAS_NUM_THREADS", "1")
    os.environ.setdefault("NUMEXPR_NUM_THREADS", "1")
    os.environ.setdefault("OMP_NUM_THREADS", "1")

    # plot_patient_variables_grid(dfx, pid)
    dfx_flagged, title_tag = detect_anomalies(dfx)            # ← one function
    plot_patient_variables_grid(dfx_flagged, pid, title_suffix=title_tag)
    return pid

In [ ]:
want = [886482, 1207865, 1452945, 611957, 965594, 5665, 7429, 29903]
tasks = [(pid, g.copy()) for pid, g in df.groupby("Patient Number") if pid in want and not g.empty]
print(f"✓ patient grouped. Total: ({len(tasks)})")

max_workers = max(1, min(2, (os.cpu_count() or 8) - 2))
with ProcessPoolExecutor(max_workers=max_workers) as ex:
    futures = [ex.submit(_plot_one, t) for t in tasks]
    for fut in as_completed(futures):
        print(f"✓ plotted patient {fut.result()}")

✓ patient grouped. Total: (8)


In [8]:
max_workers = max(1, min(24, (os.cpu_count() or 8) - 2))
max_workers

24

In [3]:
import os
os.cpu_count()

64

In [10]:
want = [886482, 1207865, 1452945, 611957, 965594, 5665, 7429, 29903]
tasks = [(pid, g.copy()) for pid, g in df.groupby("Patient Number") if pid in want and not g.empty]

In [11]:
tasks

[(5665,
         Prescription Code  Patient Number Gender  Date of Birth Visit Type  \
  209240          OPE7123G            5665    NaN            NaN        NaN   
  209241          OPE7123G            5665    NaN            NaN        NaN   
  209242          OPE7123G            5665    NaN            NaN        NaN   
  209243          OPE7123G            5665    NaN            NaN        NaN   
  209244          OPE7123G            5665    NaN            NaN        NaN   
  209245          OPE7123G            5665    NaN            NaN        NaN   
  209246          OPE7123G            5665    NaN            NaN        NaN   
  209247          OPF6002G            5665    NaN            NaN        NaN   
  209248          OPF6002G            5665    NaN            NaN        NaN   
  209249          OPF6002G            5665    NaN            NaN        NaN   
  209250          OPF6002G            5665    NaN            NaN        NaN   
  209251          OPF6002G            5665  

In [34]:
plot_patient_variables_grid(dfx_flagged, pid, title_suffix=title_tag)

In [70]:
dfx_flagged.columns

Index(['Prescription Code', 'Patient Number', 'Gender', 'Date of Birth',
       'Visit Type', 'Treatment Date', 'Prescription Name',
       'Prescription Date', 'Implementation Date', 'Result item name',
       'Result Numerical Value', 'Laboratory', 'Implementation laboratory',
       'Region', 'Result Value', 'Pacs Number', 'Variable', 'Measurement',
       'Test', 'Anomaly_Missing', 'Anomaly_Range', 'Outlier_MAD',
       'Outlier_Jump', 'Outlier', 'Anomaly_Tags', 'Anomaly_Tags_Short',
       'Anomaly_Any'],
      dtype='object')

In [74]:
dfx_flagged.head()

,Prescription Code,Patient Number,Gender,Date of Birth,Visit Type,Treatment Date,Prescription Name,Prescription Date,Implementation Date,Result item name,...,Measurement,Test,Anomaly_Missing,Anomaly_Range,Outlier_MAD,Outlier_Jump,Outlier,Anomaly_Tags,Anomaly_Tags_Short,Anomaly_Any
7855,FE7123,611957,F,19450420.0,O,20250325.0,Bronchodilator Test,2025-03-25,20250418.0,FVC_Post_Meas,...,FVC,Bronchodilator Test,False,False,False,True,True,Jump,Jump,True
7856,FE7123,611957,F,19450420.0,O,20250325.0,Bronchodilator Test,2025-03-25,20250418.0,FVC_Post_%Pred,...,FVC,Bronchodilator Test,False,False,True,True,True,MAD | Jump,MAD | Jump,True
7857,FE7123,611957,F,19450420.0,O,20250325.0,Bronchodilator Test,2025-03-25,20250418.0,FVC_Post_%Chg,...,FVC,Bronchodilator Test,False,False,False,True,True,Jump,Jump,True
7858,FE7123,611957,F,19450420.0,O,20250325.0,Bronchodilator Test,2025-03-25,20250418.0,FEV1_Post_Meas,...,FEV1,Bronchodilator Test,False,False,False,True,True,Jump,Jump,True
7859,FE7123,611957,F,19450420.0,O,20250325.0,Bronchodilator Test,2025-03-25,20250418.0,FEV1_Post_%Pred,...,FEV1,Bronchodilator Test,False,False,True,True,True,MAD | Jump,MAD | Jump,True


In [59]:
patient_ids = [886482, 1207865, 1452945, 611957, 965594, 5665, 7429, 29903]

for pid in patient_ids:
    dfx = df[df["Patient Number"] == pid].copy()
    dfx_flagged, title_tag = detect_anomalies(dfx)            # ← one function
    plot_patient_variables_grid(dfx_flagged, pid, title_suffix=title_tag)

In [60]:
pid = 611957
dfx = df[df["Patient Number"] == pid].copy()
dfx_flagged, title_tag = detect_anomalies(dfx)            # ← one function
plot_patient_variables_grid(dfx_flagged, pid, title_suffix=title_tag)

In [77]:
pid = 5665
dfx = df[df["Patient Number"] == pid].copy()
dfx_flagged, title_tag = detect_anomalies(dfx)            # ← one function
plot_patient_variables_grid(dfx_flagged, pid, title_suffix=title_tag)

In [ ]:
def data_anomalies(df):
    1. missing value if value is NaN or zero.
    then 
    def flag_outliers(
    df: pd.DataFrame,
    z_thresh: float = 3.5,
    jump_thresh_per_month: Dict[str, float] | None = None
) -> pd.DataFrame:
    import numpy as np

    jump_thresh_per_month = jump_thresh_per_month or {
        "FEV1": 0.30, "FVC": 0.40, "DLCO": 3.0, "FEV1/FVC": 0.08, "DLCO/VA": 0.60,
    }

    def _flag(g):
        g = g.sort_values("Reception Date").copy()

        # >>> THIS LINE CHANGED: safe fallback if the imputed column doesn't exist
        if "Result Numerical Value Imputed" in g.columns:
            v = g["Result Numerical Value Imputed"].fillna(g["Result Numerical Value"])
        else:
            v = g["Result Numerical Value"]

        # ensure numeric
        v = pd.to_numeric(v, errors="coerce")

        # --- MAD outlier ---
        med = v.median()
        mad = np.median(np.abs(v - med)) or 1e-9
        robust_z = 0.6745 * (v - med) / mad
        g["Outlier_MAD"] = robust_z.abs() > z_thresh

        # --- Time-aware jump outlier ---
        dv = v.diff().abs()
        dt_days = g["Reception Date"].diff().dt.days
        dt_days = dt_days.where(dt_days > 0, 1)  # avoid 0/NaN
        months = dt_days / 30.0

        metric = g["measure_index"].iat[0]
        thr_per_month = jump_thresh_per_month.get(metric, np.inf)
        dv_per_month = dv / months
        g["Outlier_Jump"] = dv_per_month > thr_per_month

        g["Outlier"] = g["Outlier_MAD"] | g["Outlier_Jump"]
        g["Outlier_Reason"] = np.where(
            g["Outlier"],
            np.where(g["Outlier_MAD"] & g["Outlier_Jump"], "MAD+Jump",
                     np.where(g["Outlier_MAD"], "MAD", "Jump")),
            ""
        )
        return g
    # convert to per patient fucntion




def _outlier_reason(row) -> str:
    mad = bool(row.get("Outlier_MAD", False))
    jmp = bool(row.get("Outlier_Jump", False))
    if mad and jmp:
        return "MAD+Jump"
    if mad:
        return "MAD"
    if jmp:
        return "Jump"
    return ""



SyntaxError: invalid syntax (644402985.py, line 2)

In [ ]:
patient_ids = [886482, 1207865, 1452945, 611957, 965594, 5665, 7429, 29903]
for pid in patient_ids:
    dfx = df[(df["Patient Number"] == pid)]
    _flag_anomaly,anomaly_type = data_anomalies(dfx)
    if _flag_anomaly:
        plot_patient_variables_grid(dfx, pid, anomaly_type)

# data anomaly

In [23]:
from pathlib import Path
from typing import Dict, Tuple
import numpy as np
import pandas as pd
from pathlib import Path
from typing import Dict
import pandas as pd
 

# FIGURES_DIR = Path("reports/figures")

def detect_anomalies(dfx: pd.DataFrame,
                     date_col: str = "Prescription Date",
                     value_col: str = "Result Numerical Value",
                     measure_col: str = "Measurement",
                     variable_col: str = "Variable",
                     z_thresh: float = 3.5,
                     jump_thresh_per_month: Dict[str, float] | None = None
                    ) -> Tuple[pd.DataFrame, str]:
    """
    Returns:
      flagged_df (same rows as dfx, with anomaly columns)
      title_tag  (e.g., 'Missing value+Out of valid range+MAD+Jump' for this patient)

    Flags:
      - Anomaly_Missing: value == 0 or NaN
      - Anomaly_Range: outside clinical ranges
          FEV1 (Meas/Post_Meas): 0.2–10.0
          FVC  (Meas/Post_Meas): 0.3–10.0
          DLCO (Meas/Post_Meas): 0.3–50
          FEV1/FVC ratio (Meas/Post_Meas): 0.2–1.2
          %Pred ( %Pred/Post_%Pred ): 0–200
      - Outlier_MAD: |robust_z| > z_thresh (per-measurement series)
      - Outlier_Jump: month-normalized step exceeds measurement threshold
    """
    jump_thresh_per_month = jump_thresh_per_month or {
        "FEV1": 0.30, "FVC": 0.40, "DLCO": 3.0, "FEV1/FVC": 0.08, "DLCO/VA": 0.60,
    }
    abs_vars   = {"Meas", "Post_Meas"}
    perc_vars  = {"%Pred", "Post_%Pred"}

    df2 = dfx.copy()

    # Ensure numeric + datetime for calculations
    v = pd.to_numeric(df2[value_col], errors="coerce")
    dt = pd.to_datetime(df2[date_col], errors="coerce")
    df2["_val_"] = v
    df2["_date_"] = dt

    # 1) Missing (as requested: treat 0 as missing; also NaN is missing)
    df2["Anomaly_Missing"] = v.isna() | (v == 0)

    # 2) Valid ranges
    rng_flag = pd.Series(False, index=df2.index)

    def _violate(series_mask, low, high):
        if not series_mask.any(): 
            return pd.Series(False, index=df2.index)
        s = v.where(series_mask)
        return (s < low) | (s > high)

    # Absolute measurements (Meas/Post_Meas)
    m = df2[measure_col].astype(str)
    var = df2[variable_col].astype(str)

    mask_abs = var.isin(abs_vars)
    rng_flag |= _violate(mask_abs & (m == "FEV1"), 0.2, 10.0)
    rng_flag |= _violate(mask_abs & (m == "FVC"),  0.3, 10.0)
    rng_flag |= _violate(mask_abs & (m == "DLCO"), 0.3, 50.0)

    # Ratio FEV1/FVC (absolute)
    rng_flag |= _violate(mask_abs & (m == "FEV1/FVC"), 0.2, 1.2)

    # Percent predicted
    mask_perc = var.isin(perc_vars)
    rng_flag |= _violate(mask_perc, 0.0, 200.0)

    df2["Anomaly_Range"] = rng_flag.fillna(False)

    # 3) Outliers (MAD + Jump) per measurement series
    out_mad  = pd.Series(False, index=df2.index)
    out_jump = pd.Series(False, index=df2.index)

    for mi, g in df2.groupby(measure_col, dropna=False):
        g = g.sort_values("_date_")
        vv = g["_val_"]
        dd = g["_date_"]

        # MAD
        med = vv.median()
        mad = float(np.median(np.abs(vv - med))) if len(vv) else 0.0
        mad = mad if mad > 0 else 1e-9
        robust_z = 0.6745 * (vv - med) / mad
        out_mad.loc[g.index] = robust_z.abs() > z_thresh

        # Jump per month
        dv = vv.diff().abs()
        dt_days = dd.diff().dt.days
        dt_days = dt_days.where(dt_days > 0, 1)  # avoid 0/NaN/<=0
        months = dt_days / 30.0
        thr = jump_thresh_per_month.get(str(mi), np.inf)
        out_jump.loc[g.index] = (dv / months) > thr

    df2["Outlier_MAD"]  = out_mad.fillna(False)
    df2["Outlier_Jump"] = out_jump.fillna(False)
    df2["Outlier"]      = df2["Outlier_MAD"] | df2["Outlier_Jump"]

    # Compose row-wise tags
    def _row_tags(row):
        tags = []
        if row["Anomaly_Missing"]: tags.append("Missing value")
        if row["Anomaly_Range"]:   tags.append("Out of valid range")
        if row["Outlier_MAD"]:     tags.append("MAD")
        if row["Outlier_Jump"]:    tags.append("Jump")
        return " | ".join(tags)

    df2["Anomaly_Tags"] = df2.apply(_row_tags, axis=1)
    # Short tag for tight annotations
    df2["Anomaly_Tags_Short"] = (df2["Anomaly_Tags"]
                                 .str.replace("Missing value", "Miss", regex=False)
                                 .str.replace("Out of valid range", "Range", regex=False))

    df2["Anomaly_Any"] = df2[["Anomaly_Missing","Anomaly_Range","Outlier_MAD","Outlier_Jump"]].any(axis=1)

    # Build patient-level title tag
    present = []
    anomaly_flag = df2["Anomaly_Any"].any()
    if df2["Anomaly_Missing"].any(): present.append("Missing value")
    if df2["Anomaly_Range"].any():   present.append("Out of valid range")
    if df2["Outlier_MAD"].any():     present.append("MAD")
    if df2["Outlier_Jump"].any():    present.append("Jump")
    title_tag = "+".join(present)

    # Clean temp cols for plotting (keep dates numeric as original)
    df2 = df2.drop(columns=["_val_", "_date_"])

    return df2, title_tag, anomaly_flag

In [41]:
df = pd.read_csv(Path(r"D:\Research\Project_COPD\COPD\data\interim\ALL_PRESCRIPTION_DATA_FILTERED.csv"))
df["Prescription Date"] = pd.to_datetime(df["Prescription Date"], format="%Y%m%d", errors="coerce")
df["Prescription Date"] = sorted(df["Prescription Date"])
df["Result Numerical Value"] = pd.to_numeric(df["Result Numerical Value"], errors="coerce")

pid = 29903

dfx = df[df["Patient Number"] == pid]
dfx_flagged, title_tag, anomaly_flag = detect_anomalies(dfx)

C:\Users\Shayahn-DKE\AppData\Local\Temp\ipykernel_57024\1593342354.py:1: DtypeWarning: Columns (2,4,10,11,12,13,14) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(Path(r"D:\Research\Project_COPD\COPD\data\interim\ALL_PRESCRIPTION_DATA_FILTERED.csv"))


In [38]:
dfx_flagged.columns

Index(['Prescription Code', 'Patient Number', 'Gender', 'Date of Birth',
       'Visit Type', 'Treatment Date', 'Prescription Name',
       'Prescription Date', 'Implementation Date', 'Result item name',
       'Result Numerical Value', 'Laboratory', 'Implementation laboratory',
       'Region', 'Result Value', 'Pacs Number', 'Variable', 'Measurement',
       'Test', 'Anomaly_Missing', 'Anomaly_Range', 'Outlier_MAD',
       'Outlier_Jump', 'Outlier', 'Anomaly_Tags', 'Anomaly_Tags_Short',
       'Anomaly_Any'],
      dtype='object')

In [42]:
sorted(dfx["Measurement"].dropna().unique().tolist())

['FEV1', 'FEV1/FVC', 'FVC']

In [40]:
desired_columns = ['Patient Number', 'Prescription Date', 'Test', 'Measurement', 'Variable', 'Result Numerical Value', 'Anomaly_Missing', 'Anomaly_Range', 'Outlier_MAD',
       'Outlier_Jump', 'Outlier', 'Anomaly_Tags', 'Anomaly_Tags_Short', 'Anomaly_Any']
dfx_flagged[desired_columns].tail()

,Patient Number,Prescription Date,Test,Measurement,Variable,Result Numerical Value,Anomaly_Missing,Anomaly_Range,Outlier_MAD,Outlier_Jump,Outlier,Anomaly_Tags,Anomaly_Tags_Short,Anomaly_Any
683497,29903,2023-02-20,PFT,FEV1/FVC,Meas,65.26,False,True,True,True,True,Out of valid range | MAD | Jump,Range | MAD | Jump,True
683498,29903,2023-02-20,PFT,FEV1/FVC,Pred,0.00,True,False,False,True,True,Missing value | Jump,Miss | Jump,True
683499,29903,2023-02-20,PFT,FVC,%Pred,89.08,False,False,True,True,True,MAD | Jump,MAD | Jump,True
683500,29903,2023-02-20,PFT,FVC,Meas,2.06,False,False,False,True,True,Jump,Jump,True
683501,29903,2023-02-20,PFT,FVC,Pred,2.31,False,False,False,True,True,Jump,Jump,True


In [5]:
df.columns

Index(['Prescription Code', 'Patient Number', 'Gender', 'Date of Birth',
       'Visit Type', 'Treatment Date', 'Prescription Name',
       'Prescription Date', 'Implementation Date', 'Result item name',
       'Result Numerical Value', 'Laboratory', 'Implementation laboratory',
       'Region', 'Result Value', 'Pacs Number', 'Variable', 'Measurement',
       'Test'],
      dtype='object')

In [10]:
uni_v =df["Variable"].unique()
uni_v

array(['Meas', '%Pred', 'Post_Meas', 'Post_%Pred', 'Post_%Chg', 'Pred',
       '%Chg.'], dtype=object)

In [21]:
v_col = "Result Numerical Value"
for v in uni_v:
    dfx_v = df[df["Variable"] == v]
    print(f"Variable: {v}, min: {dfx_v[v_col].min():.1f}, 2nd min: {dfx_v[v_col].nsmallest(2).iloc[-1]:.1f}, max: {dfx_v[v_col].max():.1f}, mean: {dfx_v[v_col].mean():.1f}")

Variable: Meas, min: 0.0, 2nd min: 0.0, max: 100.0, mean: 22.8
Variable: %Pred, min: -226.8, 2nd min: -191.7, max: 896.0, mean: 74.2
Variable: Post_Meas, min: 0.4, 2nd min: 0.4, max: 7.7, mean: 2.8
Variable: Post_%Pred, min: 19.0, 2nd min: 20.0, max: 233.0, mean: 97.5
Variable: Post_%Chg, min: -10.0, 2nd min: -10.0, max: 112.0, mean: 3.4
Variable: Pred, min: -1.2, 2nd min: -1.1, max: 86.0, mean: 18.5
Variable: %Chg., min: -34.0, 2nd min: -20.2, max: 205.0, mean: 2.7


In [27]:
#plot meas and %Pred
import matplotlib.pyplot as plt



# Create a figure with subplots
fig, axs = plt.subplots(nrows=len(uni_v), ncols=1, figsize=(10, 5 * len(uni_v)))

for ax, v in zip(axs, uni_v):
    dfx_v = df[df["Variable"] == v]
    ax.plot(dfx_v["Measurement"], dfx_v["Result Numerical Value"], label="Measured")
    ax.plot(dfx_v["Measurement"], dfx_v["%Pred"], label="%Pred")
    ax.set_title(f"Variable: {v}")
    ax.set_xlabel("Measurement")
    ax.set_ylabel("Value")
    ax.legend()

plt.tight_layout()
plt.show()

KeyError: '%Pred'

Error in callback <function _draw_all_if_interactive at 0x000001AA075B95A0> (for post_execute), with arguments args (),kwargs {}:


OverflowError: Exceeded cell block limit in Agg.  Please set the value of rcParams['agg.path.chunksize'], (currently 0) to be greater than 100 or increase the path simplification threshold(rcParams['path.simplify_threshold'] = 0.111111111111 by default and path.simplify_threshold = 0.111111111111 on the input).

OverflowError: Exceeded cell block limit in Agg.  Please set the value of rcParams['agg.path.chunksize'], (currently 0) to be greater than 100 or increase the path simplification threshold(rcParams['path.simplify_threshold'] = 0.111111111111 by default and path.simplify_threshold = 0.111111111111 on the input).

<Figure size 1000x3500 with 7 Axes>

In [8]:
df[df["Measurement"]==uni_m[0]].head()

,Prescription Code,Patient Number,Gender,Date of Birth,Visit Type,Treatment Date,Prescription Name,Prescription Date,Implementation Date,Result item name,Result Numerical Value,Laboratory,Implementation laboratory,Region,Result Value,Pacs Number,Variable,Measurement,Test
0,FE7123,5805,M,19350624.0,O,20240329.0,Bronchodilator Test,2024-03-29,20240418.0,FVC_Meas,2.26,NaN,NaN,NaN,NaN,NaN,Meas,FVC,Bronchodilator Test
1,FE7123,5805,M,19350624.0,O,20240329.0,Bronchodilator Test,2024-03-29,20240418.0,FVC_%Pred,75.00,NaN,NaN,NaN,NaN,NaN,%Pred,FVC,Bronchodilator Test
2,FE7123,5805,M,19350624.0,O,20240329.0,Bronchodilator Test,2024-03-29,20240418.0,FVC_Post_Meas,2.35,NaN,NaN,NaN,NaN,NaN,Post_Meas,FVC,Bronchodilator Test
3,FE7123,5805,M,19350624.0,O,20240329.0,Bronchodilator Test,2024-03-29,20240418.0,FVC_Post_%Pred,79.00,NaN,NaN,NaN,NaN,NaN,Post_%Pred,FVC,Bronchodilator Test
4,FE7123,5805,M,19350624.0,O,20240329.0,Bronchodilator Test,2024-03-29,20240418.0,FVC_Post_%Chg,4.00,NaN,NaN,NaN,NaN,NaN,Post_%Chg,FVC,Bronchodilator Test


In [9]:
date_col: str = "Prescription Date"
value_col: str = "Result Numerical Value"
measure_col: str = "Measurement"
variable_col: str = "Variable"
z_thresh: float = 3.5
jump_thresh_per_month = None

jump_thresh_per_month = jump_thresh_per_month or {
        "FEV1": 0.30, "FVC": 0.40, "DLCO": 3.0, "FEV1/FVC": 0.08, "DLCO/VA": 0.60,
    }
abs_vars   = {"Meas", "Post_Meas"}
perc_vars  = {"%Pred", "Post_%Pred"}

df2 = dfx.copy()

In [21]:
v = pd.to_numeric(df2[value_col], errors="coerce")
dt = pd.to_datetime(df2[date_col], errors="coerce")
df2["_val_"] = v
df2["_date_"] = dt

# 1) Missing (as requested: treat 0 as missing; also NaN is missing)
df2["Anomaly_Missing"] = v.isna() | (v == 0)

# 2) Valid ranges
rng_flag = pd.Series(False, index=df2.index)

In [24]:
m = df2[measure_col].astype(str)
m

676197        FEV1
676198        FEV1
676199        FEV1
676200    FEV1/FVC
676201    FEV1/FVC
676202    FEV1/FVC
676203         FVC
676204         FVC
676205         FVC
683484        FEV1
683485        FEV1
683486        FEV1
683487    FEV1/FVC
683488    FEV1/FVC
683489    FEV1/FVC
683490         FVC
683491         FVC
683492         FVC
683493        FEV1
683494        FEV1
683495        FEV1
683496    FEV1/FVC
683497    FEV1/FVC
683498    FEV1/FVC
683499         FVC
683500         FVC
683501         FVC
Name: Measurement, dtype: object

In [25]:
var = df2[variable_col].astype(str)
var

676197    %Pred
676198     Meas
676199     Pred
676200    %Pred
676201     Meas
676202     Pred
676203    %Pred
676204     Meas
676205     Pred
683484    %Chg.
683485    %Pred
683486     Meas
683487    %Chg.
683488    %Pred
683489     Meas
683490    %Chg.
683491    %Pred
683492     Meas
683493    %Pred
683494     Meas
683495     Pred
683496    %Pred
683497     Meas
683498     Pred
683499    %Pred
683500     Meas
683501     Pred
Name: Variable, dtype: object

In [26]:
 mask_abs = var.isin(abs_vars)

In [27]:
mask_abs

676197    False
676198     True
676199    False
676200    False
676201     True
676202    False
676203    False
676204     True
676205    False
683484    False
683485    False
683486     True
683487    False
683488    False
683489     True
683490    False
683491    False
683492     True
683493    False
683494     True
683495    False
683496    False
683497     True
683498    False
683499    False
683500     True
683501    False
Name: Variable, dtype: bool

In [28]:
var = df2[variable_col].astype(str)
var

676197    %Pred
676198     Meas
676199     Pred
676200    %Pred
676201     Meas
676202     Pred
676203    %Pred
676204     Meas
676205     Pred
683484    %Chg.
683485    %Pred
683486     Meas
683487    %Chg.
683488    %Pred
683489     Meas
683490    %Chg.
683491    %Pred
683492     Meas
683493    %Pred
683494     Meas
683495     Pred
683496    %Pred
683497     Meas
683498     Pred
683499    %Pred
683500     Meas
683501     Pred
Name: Variable, dtype: object

In [29]:
rng_flag

676197    False
676198    False
676199    False
676200    False
676201    False
676202    False
676203    False
676204    False
676205    False
683484    False
683485    False
683486    False
683487    False
683488    False
683489    False
683490    False
683491    False
683492    False
683493    False
683494    False
683495    False
683496    False
683497    False
683498    False
683499    False
683500    False
683501    False
dtype: bool

In [30]:
rng_flag.shape

(27,)

In [32]:
def _violate(series_mask, low, high):
        if not series_mask.any(): 
            return pd.Series(False, index=df2.index)
        s = v.where(series_mask)
        return (s < low) | (s > high)

In [33]:
rng_flag |= _violate(mask_abs & (m == "FEV1"), 0.2, 10.0)

In [35]:
rng_flag

676197    False
676198     True
676199    False
676200    False
676201    False
676202    False
676203    False
676204    False
676205    False
683484    False
683485    False
683486    False
683487    False
683488    False
683489    False
683490    False
683491    False
683492    False
683493    False
683494    False
683495    False
683496    False
683497    False
683498    False
683499    False
683500    False
683501    False
dtype: bool

In [2]:
df.columns

NameError: name 'df' is not defined